# Phương pháp Thu thập Dữ liệu Bài báo Khoa học

> **Môn học:** Nhập môn Khoa học Dữ liệu (Introduction to Data Science) — HCMUS  
> **Milestone 1:** Thu thập Dữ liệu và Đo lường Hiệu năng  
> **Tác giả:** Nhóm 04 (Nguyễn Lê Anh Tuấn - MSSV 24120153)  
> **Thời gian thực hiện:** Học kỳ 1, Năm học 2026–2027

---

## 1. Mục tiêu và Phạm vi Thu thập

Mục tiêu của Milestone 1 là xây dựng một quy trình thu thập dữ liệu khoa học tự động, mạnh mẽ và có thể tái lập, nhằm chuẩn bị tập dữ liệu bài báo phục vụ cho bài toán khai phá dữ liệu văn bản và trích xuất mạng lưới trích dẫn (Milestone 2 & 3).

### Yêu cầu cốt lõi theo đề bài:
1. **Metadata bài báo:** Thu thập đầy đủ các trường thông tin theo bảng `papers` (arXiv ID, Title, Authors, Submission Date, Revised Dates, Venue).
2. **Toàn văn mã nguồn TeX:** Tải về toàn bộ các version có sẵn của từng bài báo từ arXiv mirror (`export.arxiv.org/src/`), giải nén và giữ lại duy nhất các file `.tex` và `.bib` (loại bỏ hoàn toàn các file ảnh, PDF, và file nhị phân trung gian).
3. **Mạng lưới trích dẫn:** Khai thác danh mục tài liệu tham khảo (references) từ Semantic Scholar Graph API cho từng bài báo tương ứng.
4. **Đo lường hiệu năng:** Ghi nhận có hệ thống 4 chỉ số tài nguyên: End-to-end wall-clock time, Peak RAM, Average RAM và Disk usage delta.


---

## 2. Khảo sát và So sánh các Nguồn Dữ liệu

Để thu thập kho dữ liệu bài báo khoa học quy mô lớn, nhóm đã khảo sát 4 phương thức và nguồn cung cấp dữ liệu:

1. **arXiv API (Atom Feed 1.0):** Giao thức HTTP GET chính thức do arXiv cung cấp.
2. **OAI-PMH (Open Archives Initiative):** Giao thức thu hoạch siêu dữ liệu hàng loạt theo chuẩn Dublin Core và arXiv Raw XML.
3. **Kaggle arXiv Dataset:** Bộ dữ liệu tổng hợp đóng gói sẵn dạng snapshot trên Kaggle.
4. **Semantic Scholar Graph API:** Nền tảng đồ thị học thuật phục vụ khai thác trích dẫn sâu.

### Bảng so sánh đặc tính kỹ thuật giữa 4 nguồn:

| Tiêu chí Đánh giá | arXiv API | OAI-PMH (Endpoint Mới) | Kaggle arXiv Dataset | Semantic Scholar API |
|---|---|---|---|---|
| **Giao thức truyền tải** | HTTP REST (Atom 1.0) | HTTP OAI-PMH (XML) | File Download (JSON Lines) | HTTP REST (JSON) |
| **Giới hạn tốc độ (Rate Limit)** | 1 request / 3 giây | 1 request / 3 giây | Không giới hạn (Tải một lần) | 1 request / giây (Miễn phí) |
| **Độ trễ cập nhật** | Thời gian thực (Real-time) | Thời gian thực (Real-time) | Snapshot tĩnh (Trễ 1–3 tháng) | Trễ từ 1–7 ngày |
| **Phạm vi dữ liệu** | Siêu dữ liệu bài báo | Toàn bộ lịch sử metadata | Siêu dữ liệu + Abstract | Mạng lưới trích dẫn |
| **Hỗ trợ truy vấn theo ID** | Hỗ trợ danh sách ID | Truy vấn tuần tự qua token | Cần duyệt toàn bộ file lớn | Hỗ trợ batch tối đa 500 ID |
| **Vai trò trong đồ án** | **Nguồn chính (Primary)** | Nguồn dự phòng kiểm định | Đối sánh cross-check | **Nguồn chính cho References** |


---

## 3. Phân tích Chi tiết Từng Nguồn Dữ liệu

### 3.1. arXiv API (Atom 1.0 Feed)
- **Cơ chế hoạt động:** Truy vấn thông qua HTTP GET với tham số `id_list` hoặc `search_query`. Phản hồi trả về tài liệu XML chuẩn Atom 1.0.
- **Ưu điểm:** Nhẹ, phản hồi nhanh, hỗ trợ tìm kiếm theo ID cụ thể được giao trong danh sách `ids/<mssv>.txt`.
- **Hạn chế kỹ thuật quan trọng:**
  > **Hạn chế về `revised_dates`:** arXiv Atom feed chỉ cung cấp 2 mốc thời gian: `published` (ngày nộp bản đầu tiên $v_1$) và `updated` (ngày cập nhật bản mới nhất). Đối với các bài báo có từ 3 version trở lên ($n \ge 3$), ngày của các version trung gian ($v_2, v_3, \dots, v_{n-1}$) **hoàn toàn không có trong phản hồi Atom**.
  > Nhóm ghi nhận trường hợp này với cờ `"revised_dates_complete": false` để đảm bảo tính trung thực của dữ liệu.

### 3.2. OAI-PMH (Open Archives Initiative Protocol for Metadata Harvesting)
- **Cơ chế hoạt động:** Sử dụng các động từ (verbs) chuẩn của giao thức OAI-PMH: `ListRecords`, `GetRecord`, `Identify`, `ListSets`.
- **Cập nhật quan trọng (2025):** Endpoint cũ `export.arxiv.org/oai2` đã chính thức ngừng hoạt động. arXiv đã chuyển toàn bộ dịch vụ OAI-PMH sang máy chủ mới tại `https://oaipmh.arxiv.org/oai`.
- **Giá trị khai thác:** Metadata định dạng `arXivRaw` trong OAI-PMH chứa lịch sử chi tiết (full version history) bao gồm kích thước và ngày nộp chính xác của từng version $v_1, v_2, \dots, v_n$. Đây là giải pháp nâng cao (Module A3) để hoàn thiện các trường `revised_dates` còn thiếu.

### 3.3. Kaggle arXiv Dataset
- **Cơ chế hoạt động:** Bản dump nén dạng JSON lines (khoảng 4GB nén, 14GB giải nén) chứa metadata của hơn 2 triệu bài báo arXiv.
- **Giá trị khai thác:** Cho phép đối chiếu chéo (cross-validation) dữ liệu metadata đã crawl mà không tốn chi phí băng thông hay vi phạm giới hạn tốc độ của arXiv server.

### 3.4. Semantic Scholar Graph API
- **Cơ chế hoạt động:** Truy vấn REST API qua mã định danh `arXiv:<id>`.
  - Endpoint: `https://api.semanticscholar.org/graph/v1/paper/arXiv:{arxiv_id}?fields=references.title,references.authors,references.publicationDate,references.venue,references.year,references.externalIds`
- **Giá trị khai thác:** Cung cấp toàn bộ danh mục tài liệu được trích dẫn (references), bao gồm cả bài báo có arXiv ID và bài báo chỉ có DOI / S2 ID.


---

## 4. Chuẩn hóa Cấu trúc và Ánh xạ Trường Dữ liệu

Để đảm bảo tính nhất quán xuyên suốt đồ án, nhóm đã thống nhất hợp đồng dữ liệu chuẩn tại `docs/schema.md`.

### 4.1. Quy ước Khóa định danh Publication:
- `arxiv_id`: Mã định danh cơ sở không chứa version (ví dụ: `2310.12345` hoặc `hep-th/9901001`).
- `version`: Số nguyên chỉ phiên bản (`1`, `2`, `3`).
- `pub_key`: Tên thư mục an toàn trên hệ thống tệp (thay dấu `.` thành `-` và `/` thành `_`, ví dụ `2310-12345`, `hep-th_9901001`).

### 4.2. Bảng ánh xạ trường từ các nguồn vào bảng `papers`:

| Tên trường Chuẩn | Kiểu dữ liệu | Nguồn arXiv API / OAI-PMH | Nguồn Semantic Scholar | Ghi chú xử lý |
|---|---|---|---|---|
| `arxiv_id` | String | `id` (chuẩn hóa loại bỏ URL) | `externalIds.ArXiv` | Khóa chính của bài báo |
| `title` | String | `title` (loại bỏ xuống dòng thừa) | `title` | Chuẩn hóa khoảng trắng |
| `authors` | List[String] | Danh sách thẻ `author/name` | `authors[].name` | Chuẩn hóa danh sách tên tác giả |
| `abstract` | String | `summary` | `abstract` | Giữ nguyên nội dung tóm tắt |
| `submission_date`| String (ISO) | `published` | `publicationDate` | Mốc thời gian nộp bản đầu tiên |
| `revised_dates` | List[String] | Thẻ `updated` theo version | Không có | Lịch sử cập nhật các phiên bản |
| `primary_category`| String | `arxiv:primary_category/@term`| Không có | Phân loại chính (ví dụ: `cs.LG`) |
| `categories` | List[String] | Danh sách thẻ `category/@term` | `fieldsOfStudy` | Toàn bộ chuyên ngành phân loại |


---

## 5. Ràng buộc Giới hạn Tốc độ và Giải thuật Kháng lỗi

### 5.1. Ràng buộc cứng từ phía máy chủ:
- **Quy định của arXiv:** Bắt buộc chờ tối thiểu **3.0 giây** giữa 2 yêu cầu liên tiếp và chỉ mở **1 kết nối đồng thời** (`REQUEST_DELAY = 3.0s`). Đồng thời, bắt buộc sử dụng header `User-Agent` mang định danh rõ ràng theo quy định:

```text
User-Agent: <GroupName>/1.0 (academic course project; mailto:<email>)
```

- **Quy định của Semantic Scholar:** Giới hạn tối đa **1 request/giây** đối với tài khoản miễn phí (`S2_REQUEST_DELAY = 1.1s`), và hỗ trợ gửi batch tối đa 500 ID mỗi request khi sử dụng API key.

### 5.2. Giải thuật chờ lũy thừa kết hợp nhiễu ngẫu nhiên (Exponential Backoff with Jitter):
Khi gặp mã lỗi HTTP `429 (Too Many Requests)` hoặc các lỗi máy chủ tạm thời `5xx`, hệ thống áp dụng giải thuật chờ lũy thừa kết hợp nhiễu ngẫu nhiên (Full Jitter) để tránh xung đột đồng thời:

```text
t_wait = random(0, min(t_max, t_base * 2^retry))
```

Dưới đây là mã nguồn minh họa giải thuật xử lý retry trong pipeline:


In [ ]:
import time
import random
import requests

def fetch_with_backoff(url: str, headers: dict, max_retries: int = 4, base_delay: float = 3.0):
    """Truy vấn HTTP với cơ chế Exponential Backoff with Jitter."""
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, headers=headers, timeout=30)
            if resp.status_code == 200:
                return resp
            elif resp.status_code in (429, 500, 502, 503, 504):
                # Tính thời gian chờ lũy thừa kèm ngẫu nhiên hóa (Full Jitter)
                cap = min(60.0, base_delay * (2 ** attempt))
                sleep_time = random.uniform(base_delay, cap)
                print(f"[Retry {attempt+1}/{max_retries}] HTTP {resp.status_code}. Backing off for {sleep_time:.2f}s...")
                time.sleep(sleep_time)
            else:
                resp.raise_for_status()
        except requests.RequestException as exc:
            if attempt == max_retries - 1:
                raise
            sleep_time = random.uniform(base_delay, base_delay * (2 ** attempt))
            print(f"[Retry {attempt+1}/{max_retries}] Lỗi kết nối: {exc}. Chờ {sleep_time:.2f}s...")
            time.sleep(sleep_time)
    raise RuntimeError(f"Vượt quá số lần retry cho URL: {url}")


---

## 6. Xử lý Toàn văn Mã Nguồn TeX và Tối ưu Dung Lượng

Đề bài yêu cầu tải toàn văn mã nguồn của **tất cả các version có sẵn** và giải nén lấy nội dung `.tex` và `.bib`. Thách thức kỹ thuật lớn nhất ở bước này là dung lượng đĩa: trung bình một file `.tar.gz` chứa đầy đủ hình ảnh, biểu đồ độ phân giải cao và file trung gian có thể nặng tới 10–50 MB, khiến kho lưu trữ 5.000 bài có thể chạm ngưỡng 30–50 GB.

### 6.1. Quy trình xử lý source archive:
1. **Tải về có kiểm soát (Streaming Download):** Đọc luồng dữ liệu nhị phân trực tiếp từ HTTP response, kiểm tra Magic Bytes để nhận diện định dạng (gzip, tar, hay PDF thuần).
2. **Lọc bỏ dữ liệu nhị phân (Selective In-Memory Extraction):** Chỉ trích xuất các tệp có phần mở rộng `.tex` và `.bib`. Toàn bộ file ảnh (`.png`, `.jpg`, `.pdf`, `.eps`), file định dạng (`.sty`, `.cls`) và file biên dịch (`.bbl`, `.aux`) đều bị loại bỏ.
3. **Xử lý bản ghi rỗng (_EMPTY.json):** Nếu bài báo không có mã nguồn TeX (tác giả chỉ nộp PDF), version đó được đánh dấu bằng file `_EMPTY.json` theo đúng quy ước schema.


---

## 7. Khả năng Tái lập và Hạ tầng Ghi nhận Nhật Ký

Mọi hoạt động trong pipeline đều tuân thủ nguyên tắc **Idempotence (Tính bất biến khi chạy lại)**:
- Khi chạy lại lệnh cào hoặc giải nén, hệ thống tự động kiểm tra sự tồn tại của file đầu ra; nếu đã có và hợp lệ thì bỏ qua, không gửi request trùng lặp lên máy chủ.
- Mỗi lần thực thi đều ghi nhận nhật ký chi tiết vào các tệp CSV chuyên biệt (`logs/discovery.csv`, `logs/download.csv`, `logs/extract.csv`, `logs/references.csv`, `logs/benchmark_runs.csv`), phục vụ việc truy vết lỗi và phân tích thống kê định lượng.

## 8. Tài liệu Tham khảo Chính thức
1. [arXiv API User Manual](https://info.arxiv.org/help/api/user-manual.html) — Hướng dẫn kỹ thuật và quy tắc etiquette.
2. [Open Archives Initiative Protocol for Metadata Harvesting (OAI-PMH v2.0)](https://www.openarchives.org/OAI/openarchivesprotocol.html).
3. [Semantic Scholar Academic Graph API Documentation](https://api.semanticscholar.org/api-docs/).
